# Regrid data (mostly SMYLE and LR FOSI)
## you can find HR FOSI regridder in the HR FOSI folder

In [2]:
# packages
%load_ext autoreload
%autoreload 2
import xarray as xr 
import numpy as np  
import cftime
import copy
import scipy.stats
from scipy import signal
from functools import partial
import glob
import dask
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# SMYLE Utility functions
from SMYLEutils import io_utils as io
from SMYLEutils import calendar_utils as cal
from SMYLEutils import stat_utils as stat

## LR FOSI and CESM SMYLE

In [32]:
var = 'graze_sp_zint_100m' # diatC_zint_100m, diatChl_SURF, photoC_diat_zint_2
init = '11' # '02','05', '08', '11'
depth = 'surface' # '100m', '300m', '1000m'
time = 'monthly'

In [33]:
# FOSI
fosi =xr.open_dataset('/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.' + var + '.030601-036812.nc')[var]

In [38]:
fosi.nbytes/1e9

0.37158912

In [40]:
ds = fosi # , fosi

ds = ds.drop('ULONG')
ds = ds.drop('ULAT')

In [41]:
import xesmf as xe

from platform import python_version
print(xe.__version__)

0.6.0


In [42]:
obs = xr.open_dataset('/glade/work/smogen/SMYLE-extremes/OceanSODA-ETHZ_GRaCER_v2021a_1982-2020.nc')

In [43]:
regridder_smyle = xe.Regridder(ds, obs, 'bilinear', periodic=True)

/glade/u/home/smogen/.conda/envs/smyle-analysis/lib/python3.8/site-packages/xarray/core/dataarray.py:789: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return key in self.data


In [44]:
%%time
ds_rg = regridder_smyle(ds)

/glade/u/home/smogen/.conda/envs/smyle-analysis/lib/python3.8/site-packages/xesmf/frontend.py:522: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  dr_out = xr.apply_ufunc(


CPU times: user 180 ms, sys: 191 ms, total: 371 ms
Wall time: 411 ms


In [45]:
# LR FOSI
ds_rg.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/LR/'+ var +'.regrid.nc')